# End-to-End Inference Demo: GNN-Based BERT for Music Context Understanding
### Course: CSE425 / EEE474 / CSE715 Neural Networks Project
**Objective**: Demonstrate complete end-to-end inference combining structural GNN representations of audio with BERT contextual text representations to predict multi-label tags, emotion valence/arousal, and cross-modal attention maps.

In [ ]:
import sys
import os
sys.path.append("..")

import torch
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.audio_features import generate_synthetic_audio, extract_features, segment_audio, extract_segment_features
from src.graph_builder import build_segment_similarity_graph, to_pyg_data
from src.bert_encoder import MusicBERTClassifier, tokenize_texts
from src.gnn_model import MusicGNNEncoder
from src.fusion_model import GNNBERTFusionModel
from src.dataset import CONTEXT_TAGS

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using inference device: {device}")

## 1. Input Audio & Text Context Setup
We select an input audio track (rock / energetic) and its accompanying natural-language description.

In [ ]:
# Generate / Load sample track
sample_audio, chords = generate_synthetic_audio(duration=20.0, genre="rock", bpm=124.0)
caption = "An energetic rock track featuring driving electric guitar riffs, heavy drums, and an intense chorus."

print(f"Audio duration: {len(sample_audio) / 22050:.1f} seconds")
print(f"Underlying chords: {chords[:8]}...")
print(f"Context Caption: \"{caption}\"")

## 2. Audio Preprocessing & Music Structure Graph Construction
Resample to 22,050 Hz, extract log-mel spectrogram and 12-bin chroma, segment into 5s windows, and build segment graph.

In [ ]:
# Extract features
feats = extract_features(sample_audio, sr=22050)
segments = segment_audio(sample_audio, sr=22050, segment_duration=5.0)
seg_features = extract_segment_features(segments, sr=22050, feature_dim=32)

# Build graph
graph_dict = build_segment_similarity_graph(seg_features, tau=0.60, top_k=3)
print(f"Constructed Music Graph with {graph_dict['num_nodes']} segment nodes and {len(graph_dict['edge_index'][0])} edges.")

# Prepare PyTorch tensors
x = torch.tensor(graph_dict["x"], dtype=torch.float32).to(device)
edge_index = torch.tensor(graph_dict["edge_index"], dtype=torch.long).to(device)
edge_weight = torch.tensor(graph_dict["edge_weight"], dtype=torch.float32).to(device)
batch_idx = torch.zeros(x.size(0), dtype=torch.long).to(device)

# Tokenize caption
tok = tokenize_texts([caption], max_length=64)
input_ids = tok["input_ids"].to(device)
attention_mask = tok["attention_mask"].to(device)

## 3. End-to-End Inference via GNN-BERT Cross-Attention Fusion

In [ ]:
# Initialize fusion model
gnn = MusicGNNEncoder(in_dim=32, hidden_dim=64, out_dim=64, num_classes=len(CONTEXT_TAGS))
bert = MusicBERTClassifier(model_name="distilbert-base-uncased", num_classes=len(CONTEXT_TAGS))
model = GNNBERTFusionModel(gnn, bert, fusion_type="cross_attention", attn_dim=128, num_classes=len(CONTEXT_TAGS)).to(device)

# Load checkpoint if available
ckpt_path = "../results/checkpoints/task3_cross_attention.pt"
if os.path.exists(ckpt_path):
    model.load_state_dict(torch.load(ckpt_path, map_location=device))
    print("Loaded trained Task 3 checkpoint!")
else:
    print("Running inference with initialized model weights (or load trained weights).")

model.eval()
with torch.no_grad():
    outputs = model(x, edge_index, edge_weight, batch_idx, input_ids, attention_mask)

probs = outputs["probs"][0].cpu().numpy()
pred_val = outputs["valence"].item()
pred_aro = outputs["arousal"].item()

print("=== Inference Results ===")
print(f"Predicted Valence: {pred_val:.2f} / 9.0 (1=negative, 9=positive)")
print(f"Predicted Arousal: {pred_aro:.2f} / 9.0 (1=calm, 9=energetic)")

# Top 5 Predicted Tags
top5_idx = np.argsort(probs)[::-1][:5]
print("\nTop Predicted Context Tags:")
for rank, idx in enumerate(top5_idx, 1):
    print(f"  {rank}. {CONTEXT_TAGS[idx]}: {probs[idx] * 100:.1f}%")

## 4. Visualizing Predicted Emotion on Russell's Circumplex

In [ ]:
plt.figure(figsize=(7, 6))
plt.axvline(5.0, color='gray', linestyle='--', alpha=0.6)
plt.axhline(5.0, color='gray', linestyle='--', alpha=0.6)
plt.scatter([pred_val], [pred_aro], color='red', s=200, zorder=5, label='Predicted Emotion')
plt.xlim(1.0, 9.0)
plt.ylim(1.0, 9.0)
plt.xlabel('Valence (1-9)')
plt.ylabel('Arousal (1-9)')
plt.title('Predicted Emotion Coordinates on Valence-Arousal Space', fontweight='bold')
plt.legend(loc='lower left')
plt.grid(True, linestyle=':', alpha=0.6)
plt.show()